# Lesson 5: File Editor

Add `edit_file` so the agent can create and modify files. This completes the read-edit-verify cycle.

In [ ]:
%pip install openai-agents mermaid-py --upgrade --quiet

<div style='margin-top: 20px; margin-bottom: 20px; text-align: center; display: flex; flex-direction: column; align-items: center;'>
<img src='./images/lesson_5_file_editor.png' width='600' /> 
</div>

In [2]:
import os
import subprocess
from getpass import getpass

from agents import Agent, Runner, function_tool

MODEL = "gpt-5.4-mini"

In [3]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

In [4]:
@function_tool
def read_file(path: str) -> str:
    """Read the contents of a file at the given path."""
    try:
        with open(path, "r") as f:
            content = f.read()
        if len(content) > 10000:
            content = content[:10000] + "\n... (truncated)"
        return content
    except FileNotFoundError:
        return f"Error: File not found: {path}"
    except Exception as e:
        return f"Error reading file: {e}"


@function_tool
def list_files(path: str = ".") -> str:
    """List all files and directories at the given path. Directories end with /."""
    try:
        entries = []
        for entry in sorted(os.listdir(path)):
            full = os.path.join(path, entry)
            if os.path.isdir(full):
                entries.append(f"{entry}/")
            else:
                entries.append(entry)
        return "\n".join(entries) if entries else "(empty directory)"
    except FileNotFoundError:
        return f"Error: Directory not found: {path}"
    except Exception as e:
        return f"Error listing files: {e}"


@function_tool
def run_command(command: str) -> str:
    """Run a shell command and return its output."""
    try:
        result = subprocess.run(command, shell=True, capture_output=True, text=True, timeout=30)
        output = result.stdout
        if result.stderr:
            output += f"\nSTDERR:\n{result.stderr}"
        if result.returncode != 0:
            output += f"\n(exit code: {result.returncode})"
        if len(output) > 10000:
            output = output[:10000] + "\n... (truncated)"
        return output if output.strip() else "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: Command timed out after 30 seconds"
    except Exception as e:
        return f"Error running command: {e}"

## Defining the `edit_file` Tool

The edit tool supports three modes:
- **Create**: when `old_str` is empty and the file doesn't exist
- **Append**: when `old_str` is empty and the file exists
- **Replace**: find `old_str` and replace with `new_str` (must match exactly once)

In [5]:
@function_tool
def edit_file(path: str, old_str: str, new_str: str) -> str:
    """Edit a file by replacing old_str with new_str.
    If old_str is empty, creates the file (or appends if it exists).
    The old_str must match exactly once to avoid ambiguous edits."""
    try:
        if old_str == "":
            # Create or append mode
            os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
            mode = "a" if os.path.exists(path) else "w"
            with open(path, mode) as f:
                f.write(new_str)
            action = "Appended to" if mode == "a" else "Created"
            return f"{action} {path}"
        else:
            # Replace mode
            with open(path, "r") as f:
                content = f.read()
            count = content.count(old_str)
            if count == 0:
                return f"Error: old_str not found in {path}"
            if count > 1:
                return f"Error: old_str found {count} times in {path} (expected exactly 1)"
            content = content.replace(old_str, new_str, 1)
            with open(path, "w") as f:
                f.write(content)
            return f"Edited {path}"
    except Exception as e:
        return f"Error editing file: {e}"

In [6]:
agent = Agent(
    name="File Editor",
    instructions="You are a coding assistant that can explore, read, edit files, and run commands. After editing, verify your changes work.",
    model=MODEL,
    tools=[read_file, list_files, run_command, edit_file],
)

In [ ]:
result = await Runner.run(
    agent,
    "Create a file called hello_world.py that contains fizz buzz. Then I want you to run it and tell me the output. Also show me how many files we have in our current working directory.",
)
print(result.final_output)

Done — `fizzbuzz.py` was created and run successfully. It prints the expected FizzBuzz output for 1–20.


In [8]:
# Clean up
if os.path.exists("fizzbuzz.py"):
    os.remove("fizzbuzz.py")

## Summary

- Added `edit_file` with create, append, and replace modes.
- The agent can now do the full cycle: explore → read → edit → run → verify.
- The single-match constraint on `old_str` prevents ambiguous edits.
- Next lesson: add `search_code` for pattern-based code search.